In [1]:
%%capture
!pip install -q -U "transformers>=4.46" accelerate peft bitsandbytes trl datasets scikit-learn joblib


In [2]:
import os
import json
import random
import zipfile

import numpy as np
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"


In [3]:
DATA_DIR = "data"

kid_adult_path = os.path.join(DATA_DIR, "kid_adult.jsonl")
public_test_style_path = os.path.join(DATA_DIR, "public_test_style.jsonl")
style_clf_path = os.path.join(DATA_DIR, "style_clf.pkl")

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

kid_adult = load_jsonl(kid_adult_path)
public_test_style = load_jsonl(public_test_style_path)


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(record):
    messages = [
        {"role": "user", "content": record["prompt"]},
        {"role": "assistant", "content": record["kid"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

sft_records = [format_example(r) for r in kid_adult]
train_dataset = Dataset.from_list(sft_records)


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [6]:
sft_config = SFTConfig(
    output_dir="./sft_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    seed=SEED,
    data_seed=SEED,
    report_to=[],
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.321751
20,1.509713
30,1.358050
40,1.229855
50,1.217054
60,1.173199
70,1.178824
80,1.153663
90,1.165356
100,1.021621


TrainOutput(global_step=282, training_loss=1.0079170900879177, metrics={'train_runtime': 3716.5681, 'train_samples_per_second': 1.202, 'train_steps_per_second': 0.076, 'total_flos': 1.459239016026624e+16, 'train_loss': 1.0079170900879177, 'entropy': 0.7730033463901944, 'num_tokens': 612777.0, 'mean_token_accuracy': 0.7881513569090102, 'epoch': 3.0})

In [7]:
ADAPTER_DIR = "./sft_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)


('./sft_adapter/tokenizer_config.json',
 './sft_adapter/chat_template.jinja',
 './sft_adapter/tokenizer.json')

In [8]:
model.eval()
model.config.use_cache = True

def generate_response(prompt_text, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt_text}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

test_prompts = [r["prompt"] for r in public_test_style]
generated_responses = [generate_response(p) for p in test_prompts]


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [10]:
import joblib
from scipy.sparse import hstack

style_clf = joblib.load(style_clf_path)
vec1, vec2 = style_clf["vecs"]
estimator = style_clf["clf"]

X = hstack([vec1.transform(generated_responses), vec2.transform(generated_responses)]).tocsr()
p_simple_scores = estimator.predict_proba(X)[:, 1]
p_simple_mean = float(np.mean(p_simple_scores))

print(f"P_simple: {p_simple_mean}")


P_simple: 0.9683751570834969
